In [1]:
!pip -q install "transformers>=4.42.0" "accelerate>=0.31.0" "bitsandbytes>=0.43.0" \
                "langchain>=0.2.0" "langchain-huggingface>=0.0.3" \
                "datasets>=2.19.0" "tqdm" "pandas>=2.0.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.4 MB/s eta 0:00:00


## 데이터 불러오기

In [2]:
import json, pandas as pd, requests

RAW_URL = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/dev.json"
data = requests.get(RAW_URL).json()
df = pd.DataFrame(data)

## 모델 불러오기

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   # GPU 메모리 충분 → bf16 권장
    device_map="auto",            # Accelerate가 최적 배치(멀티 GPU 포함)
)
model.eval()

terminators = [tokenizer.eos_token_id]
try:
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot_id, int) and eot_id >= 0:
        terminators.append(eot_id)
except Exception:
    pass

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 시스템 프롬프트 & 대화 템플릿 설정

In [5]:
SYSTEM_PROMPT = (
    "You are an AI assistant tasked with solving a question based on a two-person conversation. "
    "Carefully read the dialogue, understand the context, and select the most appropriate answer. "
    "당신은 두 사람의 대화를 바탕으로 문제를 해결하는 AI 어시스턴트입니다. "
    "대화를 주의 깊게 읽고 문맥을 이해한 뒤, 가장 적절한 답을 선택하세요."
)

In [33]:
def reasoning(dialogue: str, question: str):
  input_ids = tokenizer.apply_chat_template(
      [{"role": "system", "content": SYSTEM_PROMPT},
       {"role": "user", "content": f"[대화]\n{dialogue}\n\n[문항]\n{question}\n\n 우선 위 문제를 해결하기 위한 4단계 계획을 세우고, 계획을 수행을 통해 대화 내용을 분석하세요. 중간 계획을 통해 각 선지의 참 거짓을 검증하기 위해 어떤 정보가 필요한지 분석하고,마지막 단계에서 세 선지가 각각 참인지 거짓인지 검증하세요. 세 선지 중 올바른 선지는 하나뿐임을 참고하세요"}],
      add_generation_prompt=True, return_tensors="pt").to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          input_ids,
          max_new_tokens=1500,
          eos_token_id=terminators,
          do_sample=False,
          temperature=0.0,
          pad_token_id=tokenizer.eos_token_id
    )
  reasoning_log = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
  return reasoning_log

def final_answer(dialogue: str, question: str, reasoning_log: str):
  input_ids = tokenizer.apply_chat_template(
      [{"role": "system", "content": SYSTEM_PROMPT},
       {"role": "user", "content": f"[대화]\n{dialogue}\n\n[문항]\n{question}\n\n 우선 위 문제를 해결하기 위한 4단계 계획을 세우고, 계획을 수행하세요."},
       {"role": "assistant", "content": reasoning_log},
       {"role": "user", "content": f"A, B, C 중 최종 답안을 선택하세요. 출력 형식: A 또는 B 또는 C"},
       ],
      add_generation_prompt=True, return_tensors="pt").to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          input_ids,
          max_new_tokens=10,
          eos_token_id=terminators,
          do_sample=False,
          temperature=0.0,
          pad_token_id=tokenizer.eos_token_id
    )
  answer = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
  return answer


## 예시 데이터로 모델 테스트 해보기

In [34]:
sample = df.iloc[4].to_dict()
dialogue = sample["dialogue"]
question = sample["question"]
ground_truth = sample["answer"]

print(f"[대화]\n{dialogue}\n\n[문제]{question}\n\n")
temp =reasoning(dialogue, question)
print(f"[중간추론]\n{temp}\n\n")

answer = final_answer(dialogue, question, temp)
print(f"[LLM 답]: {answer}\n")
print(f"[정답]: {ground_truth}\n")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[대화]
화자2: 가사노동ㅠ
화자1: ㅠㅠㅠ 해도해도 끝이 없는 집안일..
화자2: 마자요
화자2: 돈도안주는데
화자2: 힘들기만하고
화자1: 마자요 ㅠ 돈 좀 많이 받으면서 해야하는 노동 ,ㅠㅠ
화자1: 설거지 특히 넘 싫어용..
화자2: 식기세척기사고싶어요
화자1: 저두요~ 넘 편하고좋을것같아요
화자2: 비싼게단점
화자1: 그쵸 ,, 집에 설치할 자리도 없는..
화자1: 누가 대신 해주면좋겠어용 ㅠㅋㅋ
화자2: 누가대신ㅎㅎㅎㅎ
화자2: 저희집도좁아서
화자1: 흡 ㅠㅋㅋ 근데 또 넓은데 살면 그만큼 가사노동도 많아지겠져 ㅠ?
화자2: 그땐로봇청소기
화자1: 오 좋네용 ㅋㅋ
화자2: 저도갖고싶네여
화자1: 저두요 .. 하나 마련할까하다가도 ㅋㅋ 고민하게되네용

[문제]위 대화 이후 일어날 가능성이 가장 높은 ‘다음 사건’을 선택하세요.
A. 화자1은 자신의 로봇청소기를 중고로 팔 것이다.
B. 화자1은 로봇청소기로 청소를 시작할 것이다.
C. 화자1은 로봇청소기 최저가를 검색할 것이다.




The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[중간추론]
4단계 계획:
1. 대화 내용을 다시 읽고 주요 키워드와 문맥을 파악합니다.
2. 각 선지의 내용을 분석하고, 대화에서 언급된 정보를 바탕으로 각 선지의 참 거짓 여부를 검증합니다.
3. 중간 계획을 통해 필요한 정보를 확인하고, 각 선지의 참 거짓 여부를 다시 검증합니다.
4. 마지막으로 세 선지 중 올바른 선지를 선택합니다.

### 1단계: 대화 내용 파악
대화는 가사노동에 대한 불만과 로봇청소기 구매에 대한 관심을 주제로 합니다. 화자1은 가사노동이 너무 힘들다고 말하고, 화자2는 로봇청소기를 원합니다. 화자1은 로봇청소기가 편리하다고 생각하지만, 비용과 설치 공간이 문제라고 언급합니다.

### 2단계: 각 선지 분석
A. 화자1은 자신의 로봇청소기를 중고로 팔 것이다.
- 대화에서 화자1이 로봇청소기를 중고로 팔라는 언급이 없습니다. 화자1은 로봇청소기를 원하지만, 비용과 설치 공간이 문제라고 언급했기 때문에, 자신의 로봇청소기를 팔라는 내용은 대화에 없습니다. **거짓**

B. 화자1은 로봇청소기로 청소를 시작할 것이다.
- 대화에서 화자1이 로봇청소기로 청소를 시작할 것이라는 언급이 없습니다. 화자1은 로봇청소기가 편리하다고 생각하지만, 현재로서는 구입이 어려운 상황입니다. **거짓**

C. 화자1은 로봇청소기 최저가를 검색할 것이다.
- 대화에서 화자1이 로봇청소기 최저가를 검색할 것이라는 언급이 없습니다. 하지만 화자1은 로봇청소기가 편리하다고 생각하고, 화자2와 함께 로봇청소기를 구입하는 것을 고려하고 있습니다. **참**

### 3단계: 중간 계획 검증
- 화자1이 로봇청소기를 구입할 가능성이 높습니다. 화자1은 로봇청소기가 편리하다고 생각하고, 화자2와 함께 구입하는 것을 고려하고 있습니다.
- 화자1이 로봇청소기 최저가를 검색할 가능성이 있습니다. 화자1은 로봇청소기를 구입하려는 의사가 명확하며, 최저가를 검색하는 것이 합리적인 선택입니다.

### 4단계: 최종 검증
- 세 선지 중에서 C. 화자1은 로봇청소기 최저

## dev 데이터셋: multihop(계획→추론→최종답)으로 성능 검증 + 카테고리별 요약

In [35]:
import re, numpy as np
from tqdm import tqdm
import pandas as pd

# 정답 파싱 유틸 (그대로 사용)
def extract_choice(text: str) -> str:
    if not text:
        return ""
    m = re.search(r"\b([ABC])\b", text.strip())
    if m: return m.group(1)
    m = re.search(r"[정답답안]\s*[:：]\s*([ABC])", text)
    if m: return m.group(1)
    m = re.search(r"[（(]\s*([ABC])\s*[)）]", text)
    if m: return m.group(1)
    for c in "ABC":
        if c in text: return c
    return ""

# 카테고리 컬럼명 탐색 (없으면 None)
possible_cols = ["category", "카테고리", "type", "question_category"]
category_col = next((c for c in possible_cols if c in df.columns), None)

preds, gts, ids, cats, raws_reason, raws_answer = [], [], [], [], [], []

print("=== dev 데이터셋 multihop 평가 시작 ===")
for row in tqdm(df.to_dict(orient="records")):
    dialogue = row["dialogue"]
    question = row["question"]
    gold = row["answer"].strip()

    # 1) 계획+추론 로그 생성
    rlog = reasoning(dialogue, question)

    # 2) 최종 답 생성
    ans_text = final_answer(dialogue, question, rlog)

    # 3) 파싱
    pred = extract_choice(ans_text)
    cat = row.get(category_col, "UNKNOWN") if category_col is not None else "UNKNOWN"

    preds.append(pred)
    gts.append(gold)
    ids.append(row["id"])
    cats.append(cat)
    raws_reason.append(rlog)
    raws_answer.append(ans_text)

# 결과 프레임
res = pd.DataFrame({
    "id": ids,
    "category": cats,
    "gold": gts,
    "pred": preds,
    "correct": [int(p == g) for p, g in zip(preds, gts)],
    "reason_log": raws_reason,   # 필요시 확인
    "raw_answer": raws_answer    # 필요시 확인
})

# 카테고리별 집계
cat_order = ["후행사건", "동기", "전제", "반응"]
grp = (
    res.groupby("category", dropna=False)
       .agg(n=("correct", "size"), correct=("correct", "sum"))
       .assign(accuracy=lambda d: d["correct"] / d["n"])
)

ordered_index = [c for c in cat_order if c in grp.index] + [c for c in grp.index if c not in cat_order]
grp = grp.loc[ordered_index]

# 전체 집계
overall = pd.DataFrame({
    "n": [int(res.shape[0])],
    "correct": [int(res["correct"].sum())],
    "accuracy": [res["correct"].mean() if res.shape[0] else np.nan],
}, index=["전체"])

# 최종 표
summary_table = pd.concat([grp, overall], axis=0)

# 퍼센트 표기 버전
display_table = summary_table.copy()
display_table["accuracy"] = (display_table["accuracy"] * 100).round(2).astype(str) + "%"

print("=== 카테고리별 및 전체 정확도 (multihop) ===")
display(display_table)

# 텍스트 한 줄 요약
correct = int(res["correct"].sum())
total = int(res.shape[0])
print(f"\nOverall Accuracy: {correct}/{total} = {correct/total:.4f}")

=== dev 데이터셋 multihop 평가 시작 ===


  0%|          | 0/151 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  1%|          | 1/151 [00:22<56:34, 22.63s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  1%|▏         | 2/151 [00:48<1:01:33, 24.79s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  2%|▏ 

=== 카테고리별 및 전체 정확도 (multihop) ===


,n,correct,accuracy
후행사건,51,38,74.51%
동기,25,18,72.0%
전제,25,16,64.0%
반응,25,18,72.0%
원인,25,13,52.0%
전체,151,103,68.21%



Overall Accuracy: 103/151 = 0.6821


## 2) test 데이터셋: multihop으로 추론 → 제출 파일 저장

In [36]:
import json, requests
from tqdm import tqdm

TEST_URL = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/test.json"
OUT_JSON = "submission_multihop.json"   # 저장 경로

# A/B/C → inference_1/2/3
mapping = {"A": "inference_1", "B": "inference_2", "C": "inference_3"}

def pick_choice(text: str):
    t = (text or "").strip().upper()
    for ch in ("A", "B", "C"):
        if ch in t:
            return ch
    if t in ("A", "B", "C"):
        return t
    return None

# 로드
test_data = requests.get(TEST_URL).json()
test_df = pd.DataFrame(test_data)
print("Loaded test samples:", len(test_df))

results = []
print("=== test 데이터셋 multihop 추론 시작 ===")
for row in tqdm(test_df.to_dict(orient="records"), desc="Multihop Inference", unit="sample"):
    dialogue = row["dialogue"]
    question = row["question"]

    # 1) 계획+추론 로그
    rlog = reasoning(dialogue, question)

    # 2) 최종 답
    ans_text = final_answer(dialogue, question, rlog)

    ch = pick_choice(ans_text)
    if ch is None:
        tqdm.write(f"[경고] {row['id']} → 응답 파싱 실패: '{ans_text}'")
        continue

    results.append({
        "id": row["id"],
        "output": mapping[ch]
    })

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"총 {len(results)}개 결과 저장 완료 → {OUT_JSON}")


Loaded test samples: 605
=== test 데이터셋 multihop 추론 시작 ===


Multihop Inference:   0%|          | 0/605 [00:00<?, ?sample/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Multihop Inference:   0%|          | 0/605 [00:27<?, ?sample/s]


KeyboardInterrupt: 

## 결과 집계

In [ ]:
# 결과 프레임
res = pd.DataFrame({
    "id": ids,
    "category": cats,
    "gold": gts,
    "pred": preds,
    "correct": [int(p == g) for p, g in zip(preds, gts)],
    "raw": raws,  # 원문 응답 확인용
})

# 카테고리별 집계
cat_order = ["후행사건", "동기", "전제", "반응"]
grp = (
    res.groupby("category", dropna=False)
       .agg(n=("correct", "size"), correct=("correct", "sum"))
       .assign(accuracy=lambda d: d["correct"] / d["n"])
)

# 보기 좋은 순서로 재정렬 (데이터에 존재하지 않으면 자동으로 스킵)
ordered_index = [c for c in cat_order if c in grp.index] + [c for c in grp.index if c not in cat_order]
grp = grp.loc[ordered_index]

# 전체 집계
overall = pd.DataFrame({
    "n": [int(res.shape[0])],
    "correct": [int(res["correct"].sum())],
    "accuracy": [res["correct"].mean() if res.shape[0] else np.nan],
}, index=["전체"])

# 최종 표
summary_table = pd.concat([grp, overall], axis=0)

# 퍼센트 포맷 버전(표시용)
display_table = summary_table.copy()
display_table["accuracy"] = (display_table["accuracy"] * 100).round(2).astype(str) + "%"

print("=== 카테고리별 및 전체 정확도 ===")
display(display_table)

# 텍스트로도 한 줄 출력
correct = int(res["correct"].sum())
total = int(res.shape[0])
print(f"\nOverall Accuracy: {correct}/{total} = {correct/total:.4f}")

=== 카테고리별 및 전체 정확도 ===


,n,correct,accuracy
후행사건,51,37,72.55%
동기,25,21,84.0%
전제,25,20,80.0%
반응,25,17,68.0%
원인,25,17,68.0%
전체,151,112,74.17%



Overall Accuracy: 112/151 = 0.7417
